## Tests — Analytics (rule-based MCC mapping)

Lightweight unit tests for `model/analytics.ipynb` mapping logic.

This notebook loads the helper cell from `model/analytics.ipynb` so there is a single source of truth for the mapping rules.

## Imports

In [3]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any


## Load mapping helpers from `model/analytics.ipynb`

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    """Find project root by locating the `data/` directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a 'data/' directory")


def load_analytics_helpers() -> dict[str, Any]:
    analytics_nb = find_project_root() / "model" / "analytics.ipynb"
    nb = json.loads(analytics_nb.read_text(encoding="utf-8"))
    helper_src = None
    for cell in nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        if "def mcc_to_category" in src and "CATEGORY_RULES" in src:
            helper_src = src
            break
    if helper_src is None:
        raise RuntimeError(f"Could not find MCC mapping helper cell in {analytics_nb}")

    ns: dict[str, Any] = {"Any": Any}
    exec(helper_src, ns, ns)
    return ns


ns = load_analytics_helpers()
mcc_to_category = ns["mcc_to_category"]
is_discretionary = ns["is_discretionary"]


## Unit tests

In [5]:
# Keyword rules
assert mcc_to_category("5812", "Eating Places and Restaurants") == "Dining"
assert mcc_to_category("5411", "Grocery Stores, Supermarkets") == "Groceries"
assert mcc_to_category("4900", "Utilities - Electric, Gas, Water, Sanitary") == "Utilities"

# Prefix fallback rules
assert mcc_to_category("5814", "") == "Dining"
assert mcc_to_category("5411", "") == "Groceries"
assert mcc_to_category("4900", "") == "Utilities"

# Unknown
assert mcc_to_category(None, None) == "Other/Uncategorized"

# Discretionary flag
assert is_discretionary("Dining") is True
assert is_discretionary("Groceries") is False

print("Analytics tests: PASS")


Analytics tests: PASS
